# Когерентность сейсмического разреза: semblance и eigenstructure

Ноутбук сравнивает два атрибута когерентности на нескольких фрагментах SEG-Y. Низкая когерентность может соответствовать разрывным нарушениям, но также возникает на шуме, границах профилей и других участках резкой смены волнового поля.

## 1. Подготовка окружения

SEG-Y в репозиторий не включён. По умолчанию ожидается файл
`MyDrive/cw_data/TrainingData_Image.segy` на Google Drive.
Измени `SEGY_PATH`, если файл расположен в другом месте.

In [ ]:
# Для Google Colab. При локальном запуске эту ячейку можно пропустить.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

!pip -q install segyio

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import segyio
from scipy.ndimage import uniform_filter1d

SEGY_PATH = Path('/content/drive/MyDrive/cw_data/TrainingData_Image.segy')
N_EDGE_TRACES = 700
N_MIDDLE_TRACES = 400
WINDOW_TRACES = 5
WINDOW_SAMPLES = 11
EPS = np.finfo(np.float32).eps

if not SEGY_PATH.exists():
    raise FileNotFoundError(
        f'SEG-Y не найден: {SEGY_PATH}. '
        'Проверь путь или смонтируй Google Drive.'
    )

## 2. Загрузка фрагментов разреза

In [ ]:
def read_trace_block(segy, start: int, stop: int) -> np.ndarray:
    """Читает диапазон трасс и возвращает массив (trace, sample)."""
    start = max(0, start)
    stop = min(stop, len(segy.trace))
    if start >= stop:
        raise ValueError(f'Пустой диапазон трасс: [{start}, {stop})')
    return np.stack([np.asarray(segy.trace[i], dtype=np.float32) for i in range(start, stop)])


def load_fragments(path: Path) -> dict[str, np.ndarray]:
    """Загружает начало, середину и конец SEG-Y для сопоставимого анализа."""
    with segyio.open(str(path), 'r', ignore_geometry=True) as segy:
        total = len(segy.trace)
        middle = total // 2
        edge = min(N_EDGE_TRACES, total)
        middle_half = min(N_MIDDLE_TRACES // 2, middle, total - middle)

        ranges = {
            'Начало куба': (0, edge),
            'Середина куба': (middle - middle_half, middle + middle_half),
            'Конец куба': (total - edge, total),
        }
        return {name: read_trace_block(segy, start, stop) for name, (start, stop) in ranges.items()}


seismic_fragments = load_fragments(SEGY_PATH)
for name, data in seismic_fragments.items():
    print(f'{name}: {data.shape[0]} трасс × {data.shape[1]} отсчётов')

## 3. Атрибут semblance по Marfurt

Для локального окна из `N` соседних трасс semblance оценивает сходство формы сигнала.
Значение близко к 1 при согласованном волновом поле и уменьшается при нарушении непрерывности.
Вместо медленного `generic_filter` используется эквивалентная векторизованная запись через скользящие суммы.

In [ ]:
def marfurt_semblance(
    data: np.ndarray,
    trace_window: int = WINDOW_TRACES,
    sample_window: int = WINDOW_SAMPLES,
) -> np.ndarray:
    """Вычисляет 2D semblance для массива (trace, sample)."""
    if trace_window % 2 == 0 or sample_window % 2 == 0:
        raise ValueError('Размеры окон должны быть нечётными.')

    trace_sum = uniform_filter1d(
        data, size=trace_window, axis=0, mode='reflect'
    ) * trace_window
    numerator = uniform_filter1d(
        trace_sum**2, size=sample_window, axis=1, mode='reflect'
    ) * sample_window

    energy = uniform_filter1d(
        data**2, size=trace_window, axis=0, mode='reflect'
    ) * trace_window
    denominator = uniform_filter1d(
        energy, size=sample_window, axis=1, mode='reflect'
    ) * sample_window

    coherence = numerator / (trace_window * denominator + EPS)
    return np.clip(coherence, 0.0, 1.0).astype(np.float32)

## 4. Eigenstructure coherence

В каждом локальном окне строится матрица взаимной энергии соседних трасс.
Отношение максимального собственного значения к сумме собственных значений характеризует,
насколько одна согласованная структура доминирует в окне.

In [ ]:
def eigenstructure_coherence(
    data: np.ndarray,
    trace_window: int = WINDOW_TRACES,
    sample_window: int = WINDOW_SAMPLES,
) -> np.ndarray:
    """Eigenstructure coherence для массива (trace, sample)."""
    if trace_window % 2 == 0 or sample_window % 2 == 0:
        raise ValueError('Размеры окон должны быть нечётными.')

    pad_t = trace_window // 2
    pad_s = sample_window // 2
    padded = np.pad(data, ((pad_t, pad_t), (pad_s, pad_s)), mode='reflect')
    windows = np.lib.stride_tricks.sliding_window_view(
        padded, (trace_window, sample_window)
    )

    result = np.empty(data.shape, dtype=np.float32)
    # Векторизуем расчёт по времени и оставляем короткий цикл только по трассам.
    for trace_idx in range(data.shape[0]):
        block = windows[trace_idx]  # (sample, trace_window, sample_window)
        gram = block @ np.swapaxes(block, -1, -2)
        eigenvalues = np.linalg.eigvalsh(gram)
        result[trace_idx] = eigenvalues[:, -1] / (eigenvalues.sum(axis=1) + EPS)

    return np.clip(result, 0.0, 1.0)

## 5. Расчёт атрибутов

In [ ]:
semblance_results = {}
eigen_results = {}

for name, data in seismic_fragments.items():
    print(f'Обработка: {name}')
    semblance_results[name] = marfurt_semblance(data)
    eigen_results[name] = eigenstructure_coherence(data)

## 6. Визуализация

In [ ]:
def seismic_clip(data: np.ndarray, percentile: float = 95.0) -> float:
    return float(np.percentile(np.abs(data), percentile))


fig, axes = plt.subplots(len(seismic_fragments), 3, figsize=(18, 14), constrained_layout=True)

for row, (name, data) in enumerate(seismic_fragments.items()):
    vmax = seismic_clip(data)
    axes[row, 0].imshow(
        data.T, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax, origin='upper'
    )
    axes[row, 1].imshow(
        semblance_results[name].T, aspect='auto', cmap='gray', vmin=0, vmax=1, origin='upper'
    )
    axes[row, 2].imshow(
        eigen_results[name].T, aspect='auto', cmap='gray', vmin=0, vmax=1, origin='upper'
    )

    axes[row, 0].set_ylabel(f'{name}\nОтсчёт')
    for ax in axes[row]:
        ax.set_xlabel('Трасса')

axes[0, 0].set_title('Исходный разрез')
axes[0, 1].set_title('Semblance (Marfurt)')
axes[0, 2].set_title('Eigenstructure coherence')
plt.show()

## Интерпретация

Разрывное нарушение ожидаемо проявляется как локальное уменьшение когерентности, потому что соседние трассы
перестают быть согласованными. При этом **низкая когерентность сама по себе не является доказательством разлома**:
она также может быть связана с шумом, выклиниванием, резкой сменой амплитуд или границей профилей.
Поэтому результаты следует сопоставлять с исходным разрезом и другими атрибутами.